In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_retries=5,    
)

In [3]:
def chat_node(state: ChatState):
    messages=state['messages']
    response= model.invoke(messages)
    return({'messages': [response]})

In [10]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [8]:
from langchain_core.messages import HumanMessage
initial_state = {
    'messages':[HumanMessage(content="What is Capital of India?")]
}
chatbot.invoke(initial_state)['messages'][-1].content

[{'type': 'text',
  'text': 'The capital of India is **New Delhi**.',
  'extras': {'signature': 'EmAKXgFpFH0TqrTnm33P1YnlTTd570nRD4SmGbT/vv75iBh6ykcM4VvqN5BIKvlRnPKSl3CUS/GjRGig1g/eHAR9rz4XY9GObzG8DaKuK8Ix1MdGH+TZjduaKtGQ6476mdg='}}]

In [12]:
config = {'configurable': {'thread_id': '1'}}

while True:
    user_message = input('Type here: ')

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    response = chatbot.invoke(
        {'messages': [HumanMessage(content=user_message)]},
        config=config
    )
    print('AI:', response['messages'][-1].content)

AI: [{'type': 'text', 'text': 'Hello Aayush! It looks like you might have had a little typo at the beginning ("exir"), but I still recognize you. \n\nHow can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0TmAWxCKAsCs3C0arQ6DGlqYPZYdkDdCBrvbG1YjvFisbyKof62hG2JyNoKYLf1L5WgNIVdpVqEQKshBc5w3eQRWu/GXMV1i+XOkgEkh3QHNh24OtQwHt4MyA='}}]
AI: [{'type': 'text', 'text': 'Your name is Aayush Aswal! You just told me a moment ago.', 'extras': {'signature': 'EmAKXgFpFH0TXAT6LnOIIktO+g0OEkfozEDFwjtPUyX5rtzt86hzWHhu58XoMjweIhX+GjvqtojQIe5Lc5V6dapXZvXhyVeVAf4PFWJ4VJ3QQKvTYDSl3n2xDbnXCa9JtYk='}}]
